In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils
from torchvision.utils import save_image
from pytorch_fid import fid_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import random
import os

In [30]:
print(torch.cuda.is_available()) 

False


In [31]:
# set seed for reproducibility... lots of libraries used
seed = 1
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [32]:
batch_size=64
image_size = 28
channel = 1 # since MNIST is greyscale

In [33]:
# Load MNIST Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # normalize greyscale to [-1,1]
])
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [34]:
# where to store samples
output_dir = 'generated_image_grids'
os.makedirs(output_dir, exist_ok=True)

In [35]:
# Generator model, 
class Generator(nn.Module):
    def __init__(self, z_dim, channels=channel):
        super().__init__()
        self.model = nn.Sequential(
            # verify size: (batchsize,1024,4,4) yes
            nn.ConvTranspose2d(in_channels=z_dim, out_channels=1024, kernel_size=4, stride=1, padding=0),
            nn.BatchNorm2d(num_features=1024), 
            nn.LeakyReLU(0.2, inplace=True),
            # verify size: (batchsize,512,8,8) yes
            nn.ConvTranspose2d(in_channels=1024, out_channels=512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # verify size: (batchsize, 256, 16, 16)
            nn.ConvTranspose2d(in_channels=512, out_channels=256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            # verify size: (batchsize, 128, 32, 132) yes
            nn.ConvTranspose2d(in_channels=256, out_channels=128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            # verify size: (batchsize, channel, 28, 28) yes
            nn.Conv2d(in_channels=128,  out_channels=channels, kernel_size=5, stride=1, padding=0, bias=False)
            )
        self.output = nn.Tanh()
        
    def forward(self, x):
        x = self.model(x)
        return self.output(x) 

In [14]:
# Discriminator model, CNN with 3 hidden layers
class Critic(nn.Module):
    def __init__(self, channels=channel):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels=channels, out_channels=256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(num_features=256), 
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(num_features=512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(num_features=1024),
            nn.LeakyReLU(0.2, inplace=True)
            )
        self.output = nn.Sequential(
            # The output of D is no longer a probability, we do not apply sigmoid at the output of D.
            nn.Conv2d(in_channels=1024, out_channels=1, kernel_size=3, stride=1, padding=0))


    def forward(self, x):
        x = self.model(x)
        return self.output(x)

In [36]:
# gets infinite loop of (random) batches 
def batches_loop(data_loader,device):
        while True:
            for i, (images, _) in enumerate(data_loader):
                yield images.to(device)

In [37]:
def train_wgan(z_dim, channels, num_epochs, batch_size, data_loader,
               clip=0.01, n_critic=5, lr=5e-5, store_samples=True, store_loss=True, 
               early_stopping=False, device='cpu'):
    #  Initialize generator and critic
    G = Generator(z_dim, channels=channels).to(device)
    C = Critic(channels=channels).to(device)
    
    # Optimizers (RMSprop used in reference paper, adaptive stepsize)
    optimizer_G = optim.RMSprop(G.parameters(), lr=lr)
    optimizer_C = optim.RMSprop(C.parameters(), lr=lr)
    # list to store generator/discriminator loss at each epoch
    discriminator_losses = []
    generator_losses = []
    # iterate through randomized batches
    data = batches_loop(data_loader, device)
    # Training loop
    for epoch in range(num_epochs):
        # Train critic
        for c_iter in range(n_critic):
            # Train Critic
            C.zero_grad()
            # Weight clipping
            for p in C.parameters():
                p.data.clamp_(-clip, clip)
            # Real data, infinite loop of batches
            images = data.__next__()
            # Check for batch to have full batch_size, last batch may be smaller
            if (images.size()[0] != batch_size):
                continue
            # latent space variable, vector of N(0,1) rvs
            z = torch.randn(batch_size, z_dim, 1, 1, device=device)
            
            # train on real images
            # get mean and reshape tensor to be (1,0)
            c_real_loss = C(images).mean()
            #c_real_loss.backward(one)
            
            # train on fake images
            fake_images = G(z)
            
            # get mean and reshape tensor to be (1,0)
            c_fake_loss = C(fake_images).mean()
            #c_fake_loss.backward(minus_one)
            
            c_loss = c_fake_loss - c_real_loss
            c_loss.backward()
            optimizer_C.step()
            if store_loss:
                discriminator_losses.append(c_loss.item())
        # Train Generator
        G.zero_grad()
        
        # Generate fake data
        z = torch.randn(batch_size, z_dim,1,1, device=device)
        fake_images = G(z)
        # Generator loss, i.e. maximize discriminator loss
        G_loss = -C(fake_images).mean()
        G_loss.backward()
        optimizer_G.step()
        # store loss
        if store_loss:
            generator_losses.append(G_loss.item())
        # Print losses for monitoring
        if epoch % 200 == 0:
            print(f"Epoch [{epoch}/{num_epochs}]  c_loss: {c_loss.item():.4f}  G_loss: {G_loss.item():.4f}")
        # see how the WGAN learns
        if store_samples and epoch in [0,int(np.floor(num_epochs/4))-1, 
                                       int(np.floor(num_epochs/2))-1, 
                                       3*int(np.floor(num_epochs/4))-1, 
                                       num_epochs-1]:
            G.eval() # set generator to evaluation mode
            with torch.no_grad():
                see = torch.randn(batch_size, z_dim,1,1, device=device)
                fake_images = G(see).detach().cpu()
                grid = vutils.make_grid(fake_images, nrow=8, normalize=True)
                grid_path = os.path.join(output_dir, f'epoch_grids/wgan_epoch_{epoch+1}.png')
                os.makedirs(os.path.dirname(grid_path), exist_ok=True)
                vutils.save_image(grid, grid_path)
                print(f'Saved generated images for epoch {epoch+1} at {grid_path}')
            G.train()  # Set back to training mode if necessary
    return G,C, {"G":generator_losses,"C":discriminator_losses}

In [26]:
# pretty much the same as the code chunk above, just use BCE loss instead
def train_dcgan(z_dim, channels, num_epochs, batch_size, data_loader,
               clip=0.01, n_critic=5, lr=5e-5, store_samples=True, store_loss=True, 
               early_stopping=False, device='cpu'):
    #  Initialize generator and critic
    G = Generator(z_dim, channels=channels).to(device)
    C = Critic(channels=channels).to(device)
    
    # Optimizers (RMSprop used in reference paper, adaptive stepsize)
    optimizer_G = optim.RMSprop(G.parameters(), lr=lr)
    optimizer_C = optim.RMSprop(C.parameters(), lr=lr)
    # list to store generator/discriminator loss at each epoch
    discriminator_losses = []
    generator_losses = []
    # iterate through randomized batches
    data = batches_loop(data_loader, device)
    # Use BCE loss
    loss = nn.BCELoss()
    # Training loop
    for epoch in range(num_epochs):
        # Train critic
        for c_iter in range(n_critic):
            # Train Critic
            # Real data, infinite loop of batches
            images = data.__next__()
            # Check for batch to have full batch_size, last batch may be smaller
            if (images.size()[0] != batch_size):
                continue
            # latent space variable, vector of N(0,1) rvs
            z = torch.randn(batch_size, z_dim, 1, 1, device=device)
            # make labels for real and fake images
            real_labels = torch.ones(batch_size, device=device)
            fake_labels = torch.zeros(batch_size, device=device)
            # train on real images
            outputs = torch.sigmoid(C(images))
            c_real_loss = loss(outputs.flatten(), real_labels)
            
            # train on fake images
            fake_images = G(z)
            outputs = torch.sigmoid(C(fake_images))
            c_fake_loss = loss(outputs.flatten(), fake_labels)

            # Optimize discriminator
            c_loss = c_real_loss + c_fake_loss
            C.zero_grad()
            c_loss.backward()
            optimizer_C.step()
            if store_loss:
                discriminator_losses.append(c_loss.item())
        # Train Generator
        G.zero_grad()
        # Generate fake data
        z = torch.randn(batch_size, z_dim,1,1, device=device)
        fake_images = G(z)
        outputs = torch.sigmoid(C(fake_images))
        # use real labels here
        g_loss = loss(outputs.flatten(), real_labels)
        g_loss.backward()
        optimizer_G.step()
        # store loss
        if store_loss:
            generator_losses.append(g_loss.item())
        # Print losses for monitoring
        if epoch % 200 == 0:
            print(f"Epoch [{epoch}/{num_epochs}]  c_loss: {c_loss.item():.4f}  g_loss: {g_loss.item():.4f}")
        # see how the WGAN learns
        if store_samples and epoch in [0,int(np.floor(num_epochs/4))-1, 
                                       int(np.floor(num_epochs/2))-1, 
                                       3*int(np.floor(num_epochs/4))-1, 
                                       num_epochs-1]:
            G.eval() # set generator to evaluation mode
            with torch.no_grad():
                see = torch.randn(batch_size, z_dim,1,1, device=device)
                fake_images = G(see).detach().cpu()
                grid = vutils.make_grid(fake_images, nrow=8, normalize=True)
                grid_path = os.path.join(output_dir, f'epoch_grids/dcgan_epoch_{epoch+1}.png')
                os.makedirs(os.path.dirname(grid_path), exist_ok=True)
                vutils.save_image(grid, grid_path)
                print(f'Saved generated images for epoch {epoch+1} at {grid_path}')
            G.train()  # Set back to training mode if necessary
    return G,C, {"G":generator_losses,"C":discriminator_losses}

In [38]:
# train wgan
# Hyperparameters based on reference paper
G_w,D_w, losses_w =  train_wgan(z_dim=100, channels=1, num_epochs=4000, batch_size=64, 
                                data_loader = dataloader, clip=0.01, n_critic=5, lr=5e-5, device="cuda")

AssertionError: Torch not compiled with CUDA enabled

In [39]:
# train dcgan
G_d,D_d, losses_d =  train_dcgan(z_dim=100, channels=1, num_epochs=4000, batch_size=64, 
                                data_loader = dataloader, clip=0.01, n_critic=5, lr=5e-5, device="cpu")

Epoch [0/4000]  c_loss: 0.0008  g_loss: 8.5085
Saved generated images for epoch 1 at generated_image_grids/epoch_grids/dcgan_epoch_1.png
Epoch [200/4000]  c_loss: 0.1134  g_loss: 4.1381
Epoch [400/4000]  c_loss: 0.0877  g_loss: 3.4824
Epoch [600/4000]  c_loss: 0.1530  g_loss: 2.1590
Epoch [800/4000]  c_loss: 0.1692  g_loss: 3.9732
Saved generated images for epoch 1000 at generated_image_grids/epoch_grids/dcgan_epoch_1000.png
Epoch [1000/4000]  c_loss: 0.1711  g_loss: 3.5375
Epoch [1200/4000]  c_loss: 0.1247  g_loss: 5.1856
Epoch [1400/4000]  c_loss: 0.0390  g_loss: 4.3405
Epoch [1600/4000]  c_loss: 0.1626  g_loss: 2.4894
Epoch [1800/4000]  c_loss: 0.1220  g_loss: 4.6722
Saved generated images for epoch 2000 at generated_image_grids/epoch_grids/dcgan_epoch_2000.png
Epoch [2000/4000]  c_loss: 0.1084  g_loss: 3.4783
Epoch [2200/4000]  c_loss: 0.0940  g_loss: 5.0964


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(0,4000), losses_w["G"], label="Wasserstein")
plt.title("Generator Loss (Wasserstein)")
plt.xlabel("Iteration")
plt.ylabel("Wasserstein Distance")
output_path = "w_generator_loss_plot.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(0,4000), losses_d["G"],color="darkorange")
plt.title("Generator Loss (BCE)")
plt.xlabel("Iteration")
plt.ylabel("Binary Cross Entropy")
output_path = "d_generator_loss_plot.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def interpolate_latent_vectors(generator, z_dim, n_steps, m, device='cuda', save_path='interpolation_grid.png'):
    """
    Generates multiple interpolated image sequences between random pairs of latent vectors.
    """
    generator.eval()
    # Generating the images
    with torch.no_grad():
        # generate m pairs of random latent vectors z1 and z2
        z1 = torch.randn(m, z_dim, 1, 1, device=device)  # Shape: (m, z_dim, 1, 1)
        z2 = torch.randn(m, z_dim, 1, 1, device=device)  # Shape: (m, z_dim, 1, 1)
        # create coefficients for walk along convex hull between the two vectors
        alphas = torch.linspace(0, 1, steps=n_steps, device=device).view(1, n_steps, 1, 1, 1)  # Shape: (1, n_steps, 1, 1, 1)
        # expand z1 and z2 to match alphas for broadcasting
        z1_expanded = z1.unsqueeze(1)  # Shape: (m, 1, z_dim, 1, 1)
        z2_expanded = z2.unsqueeze(1)  # Shape: (m, 1, z_dim, 1, 1)
        # interpolate between z1 and z2
        z_interpolated = z1_expanded * (1 - alphas) + z2_expanded * alphas  # Shape: (m, n_steps, z_dim, 1, 1)
        # reshape to (m * n_steps, z_dim, 1, 1) for batch processing
        z_interpolated = z_interpolated.view(m * n_steps, z_dim, 1, 1)
        # generate images from the interpolated latent vectors
        fake_images = generator(z_interpolated)  # Shape: (m * n_steps, channels, height, width)
        # normalize them to [0, 1] for visualization
        fake_images_normalized = (fake_images + 1) / 2 
    # create a grid of images for visualization
    grid = vutils.make_grid(fake_images_normalized, nrow=n_steps, padding=2, normalize=False)
    # plot the grid
    fig, ax = plt.subplots(figsize=(n_steps * 2, m * 2))  # Adjust figure size as needed
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
    ax.axis('off')
    ax.set_title('Latent Space Interpolation (Multiple Pairs)')
    plt.tight_layout()
    # save the grid
    plt.savefig(save_path)
    plt.close(fig)  # Close the figure to free memory
    print(f"Saved interpolation grid to {save_path}")
    return None

In [ ]:
interpolate_latent_vectors(G_w, 100, 8, 8, save_path='generated_image_grids/interpolation_grids/wgan_interpolation.png')

In [ ]:
interpolate_latent_vectors(G_d, 100, 8, 8, save_path='generated_image_grids/interpolation_grids/dcgan_interpolation.png')